# Herschrijven — trainingen naar de nieuwe stijl

Draai de pijplijn stap voor stap, met de mens als poort op de plek waar dat telt.

**Cellen 1, 2 en 4 doen géén API-calls**, en in cel 3 alleen de goedkope classificatie van
de vrije-tekst-aantekeningen (Haiku). Je leest in, joint, normaliseert de besluiten en
inspecteert wat het model straks precies te zien krijgt — vóórdat de dure schrijfcalls lopen.

**Voor je begint:**
1. `pip install -r requirements.txt`
2. Zet je API-key in een `.env` naast dit notebook.
3. Twee inputbestanden: het **scoresheet** (scorer-velden + handmatig ingevuld `actie_besluit`)
   en het **bronsheet** (`id` / `name` / `herschreven` / `content`).


## 1. Config

Paden en knoppen. `importlib.reload` zorgt dat je edits in de `.py`-bestanden meteen meekomen.

In [ ]:
from dotenv import load_dotenv
load_dotenv()

import importlib
import besluiten, sjabloon, rewrite_checks, rewrite_output, rewrite_trainings
for m in (besluiten, sjabloon, rewrite_checks, rewrite_output, rewrite_trainings):
    importlib.reload(m)
bes, uit, rw = besluiten, rewrite_output, rewrite_trainings

SCORED    = "Nieuwe lijst herschreven en dagen.xlsx"
SOURCE    = "/Users/hugovandenbelt/Downloads/Nieuwe lijst incl. herschreven en dagen.xlsx"
BESLUITEN = "besluiten.xlsx"
OUT_DIR   = "herschreven"

START = 0      # 0-based rijindex om mee te beginnen
N     = 1      # aantal trainingen vanaf START (None = tot het einde)

INSPECT = 5    # training_id dat je in cel 4/5/7 onder de loep neemt

## 2. Inlezen, joinen en de populatie bekijken

Controleer eerst dat beide sheets goed binnenkomen en dat elke gescoorde training een bron heeft.
De uitsplitsing laat zien hoeveel er daadwerkelijk automatisch herschreven worden.

In [ ]:
import pandas as pd

scored = rw._load_scored(SCORED)
src_by_id, cols = rw.load_source(SOURCE)
print("scoresheet:", scored.shape, "| bronkolommen:", cols)

ontbreekt = [t for t in scored["training_id"] if t not in src_by_id]
print("zonder bron:", ontbreekt or "geen")

al_herschreven = scored["herschreven"] == 1 if "herschreven" in scored else scored["training_id"] < 0
structureel    = scored["actualiteit_type"] == "structureel"
mens_nodig     = scored["menselijke_input_nodig"].astype(bool)
onbruikbaar    = scored["verdict"] == "onbruikbaar"
auto           = ~(al_herschreven | structureel | mens_nodig | onbruikbaar)

print(f"""
al herschreven (overslaan)        {int(al_herschreven.sum()):3d}
structureel -> human-queue        {int(structureel.sum()):3d}
menselijke input nodig            {int(mens_nodig.sum()):3d}
verdict onbruikbaar               {int(onbruikbaar.sum()):3d}
--------------------------------------
gaat de auto-herschrijving in     {int(auto.sum()):3d}""")

## 3. Besluiten normaliseren — de menselijke poort

`actie_besluit` heeft een vaste *structuur* (`<nr> <vrije tekst>`) maar vrije *tekst*.
Python splitst de structuur, een klein model classificeert de aantekening als
**doen / niet / mits**, en het resultaat landt in `besluiten.xlsx`.

Draai eerst de structuurcontrole (geen API). Daarna genereer je het sheet en kijk je
**alleen de regels met `bron=llm`** na. Corrigeer wat niet klopt, zet `bron` op `handmatig`
en draai de cel opnieuw — handmatige labels worden nooit overschreven.

In [ ]:
fouten = bes.check_alignment(SCORED)
assert not fouten, "los eerst de uitlijnfouten op"

In [ ]:
besluiten_df = bes.write_besluiten_sheet(SCORED, BESLUITEN, verbose=False)

na_te_kijken = besluiten_df[besluiten_df["bron"] == "llm"].sort_values("confidence")
print(f"{len(besluiten_df)} besluiten; {len(na_te_kijken)} door het model geclassificeerd\n")
pd.set_option("display.max_colwidth", 70)
na_te_kijken[["training_id", "nr", "besluit_ruw", "besluit", "confidence"]]

## 4. De briefing inspecteren — nog steeds zonder API-call

Dit is wat het model straks letterlijk krijgt. Controleer drie dingen:

- de **goedgekeurde** acties staan er, mét hun voorwaarde;
- de **afgewezen** acties staan onder NIET DOEN;
- `actualiteit_specifiek` en `actualiteit_samenvatting` staan er **niet** in — dat is
  onderbouwing van de scorer, geen besluit.

In [ ]:
b = rw.build_briefing_for_id(SCORED, SOURCE, INSPECT, besluiten_path=BESLUITEN)
briefing = rw.build_writer_user(b)

print(f"{b.titel} | persona {b.persona} | {b.dagen} dagen | "
      f"{len(b.goedgekeurd)} goedgekeurd, {len(b.afgewezen)} afgewezen\n")
print(briefing[:briefing.index("Brontekst:")])

rij = scored[scored.training_id == INSPECT].iloc[0]
for veld in ("actualiteit_specifiek", "actualiteit_samenvatting"):
    lek = str(rij[veld])[:60] in briefing
    print(f"{veld:28} lekt naar de schrijver: {lek}")

## 5. Eén training herschrijven

Vanaf hier lopen er API-calls. Schrijver → code-check → judge → route.

In [ ]:
catalog = rw.load_catalog()
client  = rw.make_client()

res = rw.rewrite_one(client, b, catalog)
print(f"status: {res.status}  {res.reden}")
print("flags:", res.flags or "geen")
print("toegepaste acties:", len(res.toegepaste_acties))

print(uit.render_markdown(res.document, res.titel) if res.document else "(geen document)")

## 6. Batch draaien

`append=True` + `skip_existing=True`: een afgebroken run hervat zonder opnieuw te betalen.
Draai je dezelfde selectie nog eens, dan levert dat 0 nieuwe rijen op.

In [ ]:
review = rw.rewrite_file(SCORED, SOURCE, OUT_DIR,
                         besluiten_path=BESLUITEN, start=START, limit=N)

review[["training_id", "titel", "status", "reden", "thin", "n_flags"]]

## 7. De output naast de bron leggen

De gegenereerde `content` heeft dezelfde sleutels als de bron, dus je kunt veld voor veld
vergelijken. Let op `days` en `certification` (ongewijzigd) en op de kop 3 in `intro`.

In [ ]:
import json
from score_trainings import parse_content

with open(f"{OUT_DIR}/trainingen/{INSPECT}.json", encoding="utf-8") as f:
    resultaat = json.load(f)

nieuw = resultaat["content"]
oud   = parse_content(src_by_id[INSPECT][cols["content"]])
print("sleutels gelijk aan de bron:", set(nieuw) == set(oud))

for kopje in sjabloon.KOPJES:
    v = nieuw.get(kopje.cms, "")
    print(f"\n{'='*70}\n{kopje.kop}  ({kopje.cms})\n{'='*70}")
    print(v if isinstance(v, str) else repr(v))

## 8. Statusverdeling en flags

Waar loopt de pijplijn vast, en waarop?

In [ ]:
from collections import Counter

print(review["status"].value_counts().to_string())
print()
losse_flags = Counter(f for regel in review["flags"] for f in str(regel).split(" | ") if f)
for flag, n in losse_flags.most_common(15):
    print(f"{n:3d}  {flag[:100]}")

## 9. Goud-corpus exporteren

De trainingen die al in de nieuwe stijl staan (`herschreven=1`) zijn referentiemateriaal om
spec en judge aan te kalibreren. Ze worden **niet** herschreven.

In [ ]:
rw.export_goud_corpus(SOURCE, OUT_DIR)